## Torch.nn.Embedding

In [1]:
import nltk

nltk.download('punkt')          # NLTK 토크나이저
nltk.download('punkt_tab')      # punkt 관련 테이블 리소스
nltk.download('stopwords')      # 불용어 목

[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\uk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\uk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\uk\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

## 사전학습된 임베딩 사용하지 않는 경우

In [2]:
sentences = [          
    'nice great best amazing',  # 긍정 문장 예시
    'stop lies',                # 부정/비판 문장 예시
    'pitiful nerd',             # 부정 문장 예시
    'excellent work',           # 긍정 문장 예시
    'supreme quality',          # 긍정 문장 예시
    'bad',                      # 부정 문장 예시
    'highly respectable'        # 긍정 문장 예시
]                               # 분류 모델에 넣을 입력 문장 리스트(list[str])
labels = [1, 0, 0, 1, 1, 0, 1]  # 각 문장에 대한 이진 라벨(1=긍정, 0=부정)

In [3]:
from nltk.tokenize import word_tokenize

tokenized_sentences = [word_tokenize(sent)for sent in sentences]
tokenized_sentences


[['nice', 'great', 'best', 'amazing'],
 ['stop', 'lies'],
 ['pitiful', 'nerd'],
 ['excellent', 'work'],
 ['supreme', 'quality'],
 ['bad'],
 ['highly', 'respectable']]

In [4]:
# 단어 사전 생성 + 정수 인코딩
from collections import Counter

tokens = [token for sent in tokenized_sentences for token in sent]  # 문장 리스트를 1차원으로 평탄화
word_countes = Counter(tokens)  # 전체 토큰의 등장 갯수
print(word_countes)

word_to_index = {word : index + 2 for index, word in enumerate(tokens)} # 토큰을 순서대로 인덱싱 (인덱스 +2)
word_to_index['<PAD>'] = 0  # 패딩 토큰 추가
word_to_index['<UNK>'] = 1  # OOV 토큰 추가
word_to_index = dict(sorted(word_to_index.items(), key = lambda x:x[1])) # 딕셔너리 정력 (인덱스 순)
print(word_to_index)

vocab_size = len(word_to_index) # 특수토큰 포함 전체 어휘
vocab_size

Counter({'nice': 1, 'great': 1, 'best': 1, 'amazing': 1, 'stop': 1, 'lies': 1, 'pitiful': 1, 'nerd': 1, 'excellent': 1, 'work': 1, 'supreme': 1, 'quality': 1, 'bad': 1, 'highly': 1, 'respectable': 1})
{'<PAD>': 0, '<UNK>': 1, 'nice': 2, 'great': 3, 'best': 4, 'amazing': 5, 'stop': 6, 'lies': 7, 'pitiful': 8, 'nerd': 9, 'excellent': 10, 'work': 11, 'supreme': 12, 'quality': 13, 'bad': 14, 'highly': 15, 'respectable': 16}


17

In [5]:
# 토큰화된 문장 리스트를 받아 단어 -> 인덱스 사전
def texts_to_sequences(sentences, word_to_index):
    sequences =[]

    for sent in sentences:  # 문장 단위 순회
        sequence = []

        for token in sent:    # 토큰 단위 순회
            if token in word_to_index:  # 사전에 있는 단어면
                sequence.append(word_to_index[token])   # 해당 단어의 값(ID) 추가
            else:
                sequence.append(word_to_index['<UNK>']) # 해당 위치에 OOV 토큰 추가

        sequences.append(sequence)
    return sequences

sequences = texts_to_sequences(tokenized_sentences, word_to_index)
sequences

[[2, 3, 4, 5], [6, 7], [8, 9], [10, 11], [12, 13], [14], [15, 16]]

In [6]:
# 패딩 추가
import numpy as np

# 서로 다른 길이의 정수 시퀀스를 0(<PAD>)으로 채워 (문장수, maxlen) 형태로 맞추는 함수
def pad_sequences(sequences, maxlen):
    # (문장수 X maxlen) 크기의 0 패딩 생성
    padded_sequences = np.zeros((len(sequences),maxlen), dtype = int)

    for index, seq in enumerate(sequences):
        # index번쨰 행에서 0번 위치부터 len(seq)-1 위치까지를 seq의 처음부터 maxlen개 까지 사용
        padded_sequences[index, : len(seq)] = seq[:maxlen]  # 앞에서부터 시퀀스 채움. 시퀀스가 길면 maxlen으로 자름
    return padded_sequences

padded_sequences = pad_sequences(sequences, maxlen = 4)

padded_sequences

array([[ 2,  3,  4,  5],
       [ 6,  7,  0,  0],
       [ 8,  9,  0,  0],
       [10, 11,  0,  0],
       [12, 13,  0,  0],
       [14,  0,  0,  0],
       [15, 16,  0,  0]])

In [7]:
padded_sequences.shape # 문장수, 고정길이4

(7, 4)

In [8]:
%pip install torch

Note: you may need to restart the kernel to use updated packages.


In [9]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class SimpleNet(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_size):
        super().__init__()

        # 단어 ID를 밀집 벡터로 변환하는 임베딩
        self.embedding = nn.Embedding(
            num_embeddings= vocab_size, # 단어 사전 크기
            embedding_dim= embedding_dim,   # 임베딩 차원
            padding_idx= 0  # 패딩 0 인덱스는 업데이트 하지않음
        )
        self.rnn = nn.RNN(embedding_dim,hidden_size, batch_first=True)  # RNN 입력(배치, 길이, 차원)
        # 마지막 은닉 상태를 받아 1차원 logit값으로 변환 (차후 BCEWithLogitsLoss 등 사용해서 확률값 변환해야함)
        self.out = nn.Linear(hidden_size,1)

    def forward(self, x):
        self.embedded = self.embedding(x)   # (batch, seq_len) -> (batch, seq_len,embedding_dim)
        out, h_n = self.rnn(self.embedded)  # out:logit값 , h_n : (num_layers*directions, batch, hidden_size)
        out = self.out(h_n.squeeze(0))
        return out

embedding_dim = 100     # 단어 벡터 차원크기
model = SimpleNet(vocab_size,embedding_dim,hidden_size=16)
model

SimpleNet(
  (embedding): Embedding(17, 100, padding_idx=0)
  (rnn): RNN(100, 16, batch_first=True)
  (out): Linear(in_features=16, out_features=1, bias=True)
)

In [10]:
%pip install torchinfo

Note: you may need to restart the kernel to use updated packages.


In [11]:
from torchinfo import summary
summary(model)  # 모델의 레이어 구성/파라미터 수 요약 정보

Layer (type:depth-idx)                   Param #
SimpleNet                                --
├─Embedding: 1-1                         1,700
├─RNN: 1-2                               1,888
├─Linear: 1-3                            17
Total params: 3,605
Trainable params: 3,605
Non-trainable params: 0

In [12]:
# 임베딩 가중치 확인 
import pandas as pd

wv = model.embedding.weight.data
print(wv.shape)

vocab = word_to_index.keys()
pd.DataFrame(wv,index=vocab)

torch.Size([17, 100])


,0,1,2,3,4,5,6,7,8,9,...,90,91,92,93,94,95,96,97,98,99
<PAD>,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
<UNK>,-1.167874,-0.116263,-0.092408,1.224528,0.266308,0.435828,-2.452903,-0.790097,-2.084724,-0.710542,...,-1.616647,1.036852,-0.700504,0.869333,-0.157151,0.150398,-1.859930,-0.911257,0.985548,1.060971
nice,-0.210509,0.210433,-1.071595,-1.324924,0.408120,0.551797,-3.428452,0.278414,-0.551793,-0.109635,...,-1.832176,-0.582993,0.040010,0.889264,-0.627155,0.238590,0.419173,-1.444033,-2.492515,-0.220528
great,-0.954398,1.518677,0.379730,-0.057676,-0.553713,-1.164676,1.121791,-0.757907,-0.492572,1.347453,...,-0.264046,-0.340439,-1.162804,2.399249,-0.730227,0.074522,1.481983,1.082434,-0.445969,0.871128
best,0.448876,0.166382,-0.142314,2.034183,0.779928,-0.371388,0.021104,-1.239378,0.288162,0.024757,...,-2.873667,0.182521,-0.497601,0.305977,1.046546,0.423288,-0.115286,-0.450324,-0.941981,-1.395009
amazing,1.249941,-1.381843,-1.015737,-0.747820,-1.013716,0.031006,-1.075554,0.415532,0.061631,-1.509216,...,0.083633,-1.272785,0.132101,-0.787255,0.933660,0.084680,0.144971,-0.054002,-1.135711,-1.044766
stop,0.466207,0.316998,1.461597,-0.935893,0.909900,-0.221078,0.894834,-0.873975,0.147236,1.837551,...,-0.694102,-0.261146,-0.004156,0.597477,0.484564,0.981723,-1.013536,0.354794,-0.516488,-0.574548
lies,-0.679049,-0.884902,1.302907,-0.631226,0.571071,-1.300802,0.704129,-0.067662,-0.392325,-1.365684,...,-0.751247,-1.274819,1.211411,-0.600769,-2.477530,0.773124,-0.886134,0.587989,-0.934213,0.606754
pitiful,-0.536837,0.102716,-0.694834,1.005638,-0.330983,-2.178417,-0.122816,-1.185243,-0.149566,-1.057567,...,-1.686976,0.736014,1.750484,-1.346813,1.017967,0.042284,-0.164551,0.186488,-2.642519,-0.231402
nerd,0.395952,-0.947576,0.761743,-0.357065,-0.840960,1.424327,-0.966519,-0.636684,0.453129,0.704747,...,0.231184,1.963835,-0.252721,1.507542,-0.195656,-0.955217,-0.011661,-2.686295,-0.716707,-0.000466


In [13]:
X = torch.tensor(padded_sequences, dtype = torch.long)
y = torch.tensor(labels, dtype = torch.float).unsqueeze(1)

dataset = TensorDataset(X,y)
dataLoader = DataLoader(dataset, batch_size=2,shuffle = True)

criterion = nn.BCEWithLogitsLoss()  # 시그모이드를 포함한 손실함수
optimizer = optim.Adam(model.parameters(),lr=0.005)

In [14]:
import torch
print(torch.__file__)


c:\Users\UK\SKN\nlp\nlp_venv\Lib\site-packages\torch\__init__.py


In [15]:
for epoch in range(20):
    epoch_loss = 0

    for X_batch, y_batch in dataLoader:
        optimizer.zero_grad()           # 이전 기울기 초기화
        output = model(X_batch)         # 순전파
        loss = criterion(output,y_batch)    # 손실 계산
        loss.backward()     # 역전파 : 기울기 계산
        optimizer.step()    # 파라미터 업데이트

        epoch_loss += loss.item()   # 미니배치 손실은 python float형태로 누적

    print(f"Epoch {epoch+1} Loss : {epoch_loss/len(dataLoader)}")

Epoch 1 Loss : 0.6724073439836502
Epoch 2 Loss : 0.5057249143719673
Epoch 3 Loss : 0.42950476706027985
Epoch 4 Loss : 0.3528597727417946
Epoch 5 Loss : 0.3109222762286663
Epoch 6 Loss : 0.21304966136813164
Epoch 7 Loss : 0.14878809452056885
Epoch 8 Loss : 0.10960593819618225
Epoch 9 Loss : 0.07820894382894039
Epoch 10 Loss : 0.057626571506261826
Epoch 11 Loss : 0.04511859826743603
Epoch 12 Loss : 0.03617646545171738
Epoch 13 Loss : 0.029077206272631884
Epoch 14 Loss : 0.02462880639359355
Epoch 15 Loss : 0.02111509209498763
Epoch 16 Loss : 0.018659634049981833
Epoch 17 Loss : 0.016976716928184032
Epoch 18 Loss : 0.014660493470728397
Epoch 19 Loss : 0.013380564516410232
Epoch 20 Loss : 0.012407018570229411


In [16]:
model.eval()

with torch.no_grad():   # 기울기 계산 비활성화
    output = model(X)   # 순전파 (예측값 생성) :
    prob = torch.sigmoid(output)    # 0~1사이 확률로 변환
    pred = (prob >= 0.5).int()      # 임계값 0.5기준으로 이진분류


print(labels)
print(pred.squeeze().detach().numpy())  # 1차원 배열 형태로 변환

[1, 0, 0, 1, 1, 0, 1]
[1 0 0 1 1 0 1]


## 사전학습된 임베딩 모델을 사용

In [17]:
from gensim.models import KeyedVectors

model_wv = KeyedVectors.load_word2vec_format('GoogleNews-vectors-negative300.bin.gz', binary = True)
model_wv.vectors.shape  # 어휘수 300000, 

(3000000, 300)

In [18]:
# 임베딩 매트릭스 초기화 후 사전학습 임베딩 차원으로 재구성

print(len(word_to_index))

embedding_matrix = np.zeros((len(word_to_index), model_wv.vectors.shape[1]))
embedding_matrix.shape

17


(17, 300)

In [19]:
# 단어가 사전학습 모델에 있으면 임베딩 벡터 반환, 없으면 None 반환
def get_word_embedding(word):
    if word in model_wv:
        return model_wv[word]
    else: 
        return None

get_word_embedding('nerd').shape

(300,)

In [20]:
for word, index in word_to_index.items():
    if index >=2:
        emb = get_word_embedding(word)
        if emb is not None:
            embedding_matrix[index] = emb

        

In [21]:
pd.DataFrame(embedding_matrix, )

,0,1,2,3,4,5,6,7,8,9,...,290,291,292,293,294,295,296,297,298,299
0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,0.158203,0.105957,-0.189453,0.386719,0.083496,-0.267578,0.083496,0.113281,-0.104004,0.178711,...,-0.085449,0.189453,-0.146484,0.134766,-0.040771,0.032715,0.089355,-0.267578,0.008362,-0.213867
3,0.071777,0.208008,-0.028442,0.178711,0.132812,-0.099609,0.096191,-0.116699,-0.008545,0.148438,...,-0.011475,0.064453,-0.289062,-0.048096,-0.199219,-0.071289,0.064453,-0.167969,-0.020874,-0.142578
4,-0.126953,0.021973,0.287109,0.153320,0.127930,0.032715,-0.115723,-0.029541,0.153320,0.011292,...,0.006439,-0.033936,-0.166016,-0.016846,-0.048584,-0.022827,-0.152344,-0.101562,-0.090332,0.088379
5,0.073730,0.004059,-0.135742,0.022095,0.180664,-0.046631,0.224609,-0.229492,-0.040039,0.225586,...,0.018433,-0.021240,-0.250000,-0.020142,-0.310547,-0.207031,-0.006317,-0.141602,-0.150391,-0.137695
6,-0.057861,0.013184,0.115234,0.069824,-0.306641,-0.044678,0.048584,0.152344,0.073242,-0.100098,...,0.100098,0.171875,-0.113281,0.064453,-0.115723,0.048096,-0.004822,0.086426,0.029907,0.007812
7,0.149414,-0.012817,0.328125,0.025513,0.017334,0.190430,0.188477,-0.143555,-0.090820,0.206055,...,-0.308594,0.183594,-0.202148,0.031494,-0.164062,-0.201172,0.080078,-0.105469,0.149414,0.157227
8,0.269531,0.253906,-0.020996,0.060303,-0.010925,0.217773,0.139648,-0.057617,0.312500,0.253906,...,-0.063477,0.132812,-0.094238,0.089355,-0.065430,-0.016235,-0.107910,-0.072266,-0.094238,0.028809
9,0.265625,-0.207031,-0.026611,0.419922,-0.208984,0.390625,0.164062,0.063965,0.149414,-0.017700,...,0.215820,0.125000,-0.227539,-0.310547,-0.112793,-0.096680,0.255859,0.124023,-0.030273,0.082031


In [22]:
# nn.Parameter를 활용해 학습가능 파라미터를 나눔 (이론)
a = torch.tensor([1.,2.,3.], requires_grad= False)
print(a.requires_grad)
b = torch.tensor([1.,2.,3.], requires_grad= True)
print(b.requires_grad)

False
True


In [23]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

class SimpleNet(nn.Module):
    def __init__(self,vocab_size,embedding_dim,hidden_size):
        super().__init__()

        # 단어 ID를 밀집 벡터로 변환하는 임베딩
        self.embedding = nn.Embedding(
            num_embeddings= vocab_size, # 단어 사전 크기
            embedding_dim= embedding_dim,   # 임베딩 차원
            padding_idx= 0  # 패딩 0 인덱스는 업데이트 하지않음
        )

        # 사전학습된 임베딩벡터로 초기
        self.embedding.weight = nn.Parameter(torch.tensor(embedding_matrix,dtype=torch.float))
        # self.embedding.weight.requires_grad = True      # True면 파인튜닝(추가학습), False면 임베딩 고정 

        self.rnn = nn.RNN(embedding_dim,hidden_size, batch_first=True)  # RNN 입력(배치, 길이, 차원)
        # 마지막 은닉 상태를 받아 1차원 logit값으로 변환 (차후 BCEWithLogitsLoss 등 사용해서 확률값 변환해야함)
        self.out = nn.Linear(hidden_size,1)

    def forward(self, x):
        self.embedded = self.embedding(x)   # (batch, seq_len) -> (batch, seq_len,embedding_dim)
        out, h_n = self.rnn(self.embedded)  # out:logit값 , h_n : (num_layers*directions, batch, hidden_size)
        out = self.out(h_n.squeeze(0))
        return out

embedding_dim = embedding_matrix.shape[1]    # 단어 벡터 차원크기
model = SimpleNet(vocab_size,embedding_dim,hidden_size=16)
model

SimpleNet(
  (embedding): Embedding(17, 300, padding_idx=0)
  (rnn): RNN(300, 16, batch_first=True)
  (out): Linear(in_features=16, out_features=1, bias=True)
)

In [24]:
print(embedding_matrix.shape)

(17, 300)


In [25]:
X = torch.tensor(padded_sequences, dtype = torch.long)
y = torch.tensor(labels, dtype = torch.float).unsqueeze(1)

dataset = TensorDataset(X,y)
dataLoader = DataLoader(dataset, batch_size=2,shuffle = True)

criterion = nn.BCEWithLogitsLoss()  # 시그모이드를 포함한 손실함수
optimizer = optim.Adam(model.parameters(),lr=0.005)

In [27]:
for epoch in range(20):
    epoch_loss = 0

    for x_batch, y_batch in dataLoader:
        optimizer.zero_grad()           # 이전 기울기 초기화
        output = model(x_batch)         # 순전파
        loss = criterion(output,y_batch)    # 손실 계산
        loss.backward()     # 역전파 : 기울기 계산
        optimizer.step()    # 파라미터 업데이트

        epoch_loss += loss.item()   # 미니배치 손실은 python float형태로 누적

    print(f"Epoch {epoch+1} Loss : {epoch_loss/len(dataLoader)}")

Epoch 1 Loss : 0.6943986564874649
Epoch 2 Loss : 0.5574987977743149
Epoch 3 Loss : 0.4505225941538811
Epoch 4 Loss : 0.3441026359796524
Epoch 5 Loss : 0.2594278044998646
Epoch 6 Loss : 0.1688133366405964
Epoch 7 Loss : 0.11583946831524372
Epoch 8 Loss : 0.08236031606793404
Epoch 9 Loss : 0.06688849162310362
Epoch 10 Loss : 0.0498158298432827
Epoch 11 Loss : 0.03946797735989094
Epoch 12 Loss : 0.03192310128360987
Epoch 13 Loss : 0.026899220887571573
Epoch 14 Loss : 0.022412225138396025
Epoch 15 Loss : 0.020229983143508434
Epoch 16 Loss : 0.016632076120004058
Epoch 17 Loss : 0.01526347384788096
Epoch 18 Loss : 0.014290979132056236
Epoch 19 Loss : 0.01278971298597753
Epoch 20 Loss : 0.011696052271872759


In [28]:
print(x_batch.shape)

torch.Size([1, 4])


In [29]:
model.eval()

with torch.no_grad():   # 기울기 계산 비활성화
    output = model(X)   # 순전파 (예측값 생성) :
    prob = torch.sigmoid(output)    # 0~1사이 확률로 변환
    pred = (prob >= 0.5).int()      # 임계값 0.5기준으로 이진분류


print(labels)
print(pred.squeeze().detach().numpy())  # 1차원 배열 형태로 변환

[1, 0, 0, 1, 1, 0, 1]
[1 0 0 1 1 0 1]


In [30]:
embedding_matrix.shape

(17, 300)